# DRF Advanced Serializers & Testing

## HyperlinkedModelSerializer in Practice

To use hyperlinks, register all related ViewSets with the router and set `extra_kwargs` to match the `view_name` the router generates:

```python
# serializers.py
from rest_framework import serializers
from .models import Author, Category, Book

class AuthorSerializer(serializers.HyperlinkedModelSerializer):
    class Meta:
        model = Author
        fields = ['url', 'id', 'name', 'slug']
        extra_kwargs = {'url': {'view_name': 'author-detail'}}

class BookSerializer(serializers.HyperlinkedModelSerializer):
    author = serializers.HyperlinkedRelatedField(
        view_name='author-detail', queryset=Author.objects.all()
    )
    categories = serializers.HyperlinkedRelatedField(
        view_name='category-detail', queryset=Category.objects.all(),
        many=True, required=False
    )
    class Meta:
        model = Book
        fields = ['url', 'id', 'title', 'author', 'categories']
        extra_kwargs = {'url': {'view_name': 'book-detail'}}
```

Example POST:
```json
{
  "title": "django in action",
  "author": "http://127.0.0.1:8000/authors/1/",
  "categories": ["http://127.0.0.1:8000/categories/1/"]
}
```


## Validation

### Field-level validation
```python
def validate_title(self, value):
    if 'django' not in value.lower():
        raise serializers.ValidationError("Title must include the word 'django'.")
    return value
```

### Object-level validation
```python
def validate(self, attrs):
    author = attrs.get('author')
    title = attrs.get('title', '')
    if author and 'anonymous' in getattr(author, 'name', '').lower() and len(title) < 5:
        raise serializers.ValidationError("Anonymous authors must use a longer title.")
    return attrs
```


## Manual Serializer Context and Image URLs

Pass extra data into a serializer using `context`:

```python
# One-off
serializer = BookSerializer(book, context={'request': request, 'is_admin': True})

# ViewSet-wide
class BookViewSet(viewsets.ModelViewSet):
    def get_serializer_context(self):
        ctx = super().get_serializer_context()
        ctx['is_admin'] = self.request.user.is_staff
        return ctx
```

Use context inside the serializer:
```python
class BookSerializer(serializers.ModelSerializer):
    is_admin_view = serializers.SerializerMethodField()
    cover_image_url = serializers.SerializerMethodField()

    def get_is_admin_view(self, obj):
        return bool(self.context.get('is_admin'))

    def get_cover_image_url(self, obj):
        request = self.context.get('request')
        if obj.cover_image and request:
            return request.build_absolute_uri(obj.cover_image.url)
        return None
```


## Test Database Lifecycle

When you run `python manage.py test`, Django:
1. Creates a separate test database (e.g. `test_mydb`) and runs migrations.
2. Wraps each `TestCase` method in a transaction and rolls it back after each test.
3. Drops the test database when finished.

### Useful flags
```bash
python manage.py test --keepdb       # reuse the test DB between runs
python manage.py test --parallel 4   # run tests in parallel
python manage.py test --failfast     # stop on the first failure
python manage.py test --verbosity 2  # more detailed output
```

### TestCase vs TransactionTestCase
- `TestCase` — wraps each test in a transaction; fast; use by default.
- `TransactionTestCase` — no transaction wrapping; use when you need to test `on_commit` hooks or committed database behavior.

### setUp vs setUpTestData
- `setUp(self)` — runs before each test method (fresh state per test).
- `@classmethod setUpTestData(cls)` — runs once per class; faster for shared read-only data.


## Django Fixtures

Fixtures are JSON (or YAML/XML) files that load data into the database.

### Creating a fixture
```bash
python manage.py dumpdata api.Author --indent 2 > api/fixtures/authors.json
```

### Fixture file format
```json
[
  {"model": "api.author", "pk": 1, "fields": {"name": "Adrian Holovaty", "slug": "adrian-holovaty"}},
  {"model": "api.author", "pk": 2, "fields": {"name": "Jacob Kaplan-Moss", "slug": "jacob-kaplan-moss"}}
]
```

### Loading fixtures in tests
```python
class AuthorTests(TestCase):
    fixtures = ['authors.json']  # auto-loaded before tests in this class

    def test_authors_exist(self):
        from api.models import Author
        self.assertEqual(Author.objects.count(), 2)
```


## API Tests with APITestCase

```python
from rest_framework.test import APITestCase
from django.urls import reverse
from api.models import Book

class BookAPITests(APITestCase):
    fixtures = ['authors.json', 'categories.json']

    def test_list_books_empty(self):
        url = reverse('book-list')
        resp = self.client.get(url)
        self.assertEqual(resp.status_code, 200)

    def test_create_book(self):
        url = reverse('book-list')
        payload = {
            "title": "django in action",
            "author": reverse('author-detail', args=[1]),
        }
        resp = self.client.post(url, payload, format='json')
        self.assertEqual(resp.status_code, 201)
        self.assertEqual(Book.objects.count(), 1)

    def test_validation_error(self):
        url = reverse('book-list')
        resp = self.client.post(url, {"title": "no keyword here", "author": reverse('author-detail', args=[1])}, format='json')
        self.assertEqual(resp.status_code, 400)
        self.assertIn('title', resp.data)
```


## Model Method Tests

```python
from django.test import TestCase
from api.models import Book, Author

class BookModelTests(TestCase):
    @classmethod
    def setUpTestData(cls):
        cls.author = Author.objects.create(name='Author', slug='author')

    def test_short_title_truncates(self):
        book = Book.objects.create(title='A very long title here', author=self.author)
        self.assertTrue(book.short_title().endswith('...'))

    def test_short_title_no_truncate(self):
        book = Book.objects.create(title='Short', author=self.author)
        self.assertEqual(book.short_title(), 'Short')
```


## Summary

- `HyperlinkedModelSerializer` uses full URLs as identifiers; set `view_name` in `extra_kwargs` to match router-generated names.
- Field-level validation uses `validate_<field>()`, object-level validation uses `validate()`.
- Use `get_serializer_context()` on a ViewSet to inject extra data (e.g. `request`, `is_admin`) into serializers.
- `fixtures = [...]` in a `TestCase` class auto-loads data before each test.
- `APITestCase.client` is DRF's `APIClient`; use `force_authenticate()` for auth in tests.
- `setUpTestData` runs once per class — faster than `setUp` when data is shared and read-only.
